# Homework Starter — Stage 04: Data Acquisition and Ingestion
Name: Wentao Zhu
Date: 2026-09-01

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/zhu/Documents/GitHub/bootcamp_wentao_zhu/homework/homework04

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

ALPHAVANTAGE_API_KEY loaded? True


## Helpers (use or modify)

In [4]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [5]:
SYMBOL = 'AAPL'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))
if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function':'TIME_SERIES_DAILY','symbol':SYMBOL,'outputsize':'compact','apikey':os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    # Alpha Vantage answers 200 OK with a prose blob when the free daily cap (25 calls)
    # is hit, so check for the series rather than trusting the status code.
    key = [k for k in js if 'Time Series' in k]
    if not key:
        print('Alpha Vantage returned no series:', str(list(js.values())[0])[:150])
        USE_ALPHA = False

if USE_ALPHA:
    df_api = pd.DataFrame(js[key[0]]).T.reset_index().rename(columns={'index':'date','4. close':'close'})[['date','close']]
    df_api['date'] = pd.to_datetime(df_api['date']); df_api['close'] = pd.to_numeric(df_api['close'])

if not USE_ALPHA:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period='3mo', interval='1d', auto_adjust=False,
                         multi_level_index=False).reset_index()[['Date','Close']]
    df_api.columns = ['date','close']

v_api = validate(df_api, ['date','close']); v_api

{'missing': [], 'shape': (100, 2), 'na_total': 0}

In [6]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

Saved data/raw/api_source-alpha_symbol-AAPL_20260901-012811.csv


## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

In [10]:
SCRAPE_URL = (
    "https://en.wikipedia.org/wiki/"
    "List_of_S%26P_500_companies"
)

request_headers = {
    "User-Agent": "NYU-Bootcamp-Homework/1.0 educational-use"
}

try:
    response = requests.get(
        SCRAPE_URL,
        headers=request_headers,
        timeout=30,
    )
    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser",
    )

    table = soup.select_one("table#constituents")

    # Fallback selector in case the table ID changes.
    if table is None:
        for candidate in soup.select("table.wikitable"):
            candidate_headers = {
                cell.get_text(strip=True)
                for cell in candidate.select("th")
            }

            if {
                "Symbol",
                "Security",
                "GICS Sector",
            }.issubset(candidate_headers):
                table = candidate
                break

    if table is None:
        raise ValueError(
            "Could not locate the S&P 500 constituents table."
        )

    column_names = [
        cell.get_text(" ",strip=True)
        for cell in table.select("tr")[0].find_all(["th", "td"])
    ]

    records = []

    for row in table.select("tr")[1:]:
        values = [
            cell.get_text(" ", strip=True)
            for cell in row.find_all(["th", "td"])
        ]

        if len(values) == len(column_names):
            records.append(values)

    df_scrape = pd.DataFrame(
        records,
        columns=column_names,
    )
except requests.RequestException as error:
    raise RuntimeError(
        f"Scraping request failed: {SCRAPE_URL}"
    ) from error
required_scrape_columns = [
    "Symbol",
    "Security",
    "GICS Sector",
    "CIK",
]

v_scrape = validate(
    df_scrape,
    required_scrape_columns,
)

for column in [
    "Symbol",
    "Security",
    "GICS Sector",
]:
    df_scrape[column] = (
        df_scrape[column]
        .astype("string")
        .str.strip()
    )

df_scrape["CIK"] = pd.to_numeric(
    df_scrape["CIK"],
    errors="coerce",
).astype("Int64")

if "Date added" in df_scrape.columns:
    df_scrape["Date added"] = pd.to_datetime(
        df_scrape["Date added"],
        errors="coerce",
    )

if df_scrape["Symbol"].isna().any():
    raise ValueError("Scraped data contains missing symbols.")

if df_scrape["CIK"].isna().any():
    raise ValueError("Scraped data contains invalid CIK values.")

print(v_scrape)
print(df_scrape.dtypes)
df_scrape.head()

{'missing': [], 'shape': (503, 8), 'na_total': 0}
Symbol                           string
Security                         string
GICS Sector                      string
GICS Sub-Industry                   str
Headquarters Location               str
Date added               datetime64[us]
CIK                               Int64
Founded                             str
dtype: object


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee , Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin , Ireland",2011-07-06,1467373,1989


In [11]:
_ = save_csv(
    df_scrape,
    prefix="scrape",
    site="wikipedia",
    table="sp500-constituents",
)

Saved data/raw/scrape_site-wikipedia_table-sp500-constituents_20260901-013956.csv


## Documentation
- API Source: (URL/endpoint/params)
- Scrape Source: (URL/table description)
- Assumptions & risks: (rate limits, selector fragility, schema changes)
- Confirm `.env` is not committed.

## Documentation

### API Source

- Source: Yahoo Finance data accessed through `yfinance`
- Ticker: `AAPL`
- Parameters:
  - `period="3mo"`
  - `interval="1d"`
  - `auto_adjust=False`
- Fields retained: `date`, `close`
- Validation:
  - DataFrame must not be empty
  - Required columns must exist
  - Dates must parse successfully
  - Close prices must be numeric and positive
  - Missing values, shape, and duplicate rows are reported

### Scrape Source

- URL: https://en.wikipedia.org/wiki/List_of_S26&P_500_companies
- Table: S&P 500 component stocks
- Primary selector: `table#constituents`
- Fallback selector: a `table.wikitable` containing the expected column names
- Validation:
  - DataFrame must not be empty
  - Required text columns must exist
  - Symbols must not be missing
  - CIK values must be numeric
  - Missing values, shape, and duplicate rows are reported

### Assumptions and Risks

- `yfinance` depends on an external data service and may be temporarily unavailable.
- Historical prices may be revised or adjusted by the data provider.
- Alpha Vantage has API rate limits and may return a successful HTTP status without returning a time series.
- Website HTML and table selectors may change, which could break the scraper.
- S&P 500 constituents change over time, so repeated runs may produce different rows.
- The data is collected for educational purposes and is not investment advice.

### Secrets

- `.env` exists only on the local machine.
- `.env` is excluded through `.gitignore`.
- `.env.example` is included in the repository without a real secret.